# Merge, join, concatenate and compare

pandas provides various methods for combining and comparing **`Series`** or **`DataFrame`**.

* **`concat()`**: Merge multiple **`Series`** or **`DataFrame`** objects along a shared index or column
* **`DataFrame.join()`**: Merge multiple **`DataFrame`** objects along the columns
* **`DataFrame.combine_first()`**: Update missing values with non-missing values in the same location
* **`merge()`**: Combine two **`Series`** or **`DataFrame`** objects with SQL-style joining
* **`merge_ordered()`**: Combine two **`Series`** or **`DataFrame`** objects along an ordered axis
* **`merge_asof()`**: Combine two **`Series`** or **`DataFrame`** objects by near instead of exact matching keys
* **`Series.compare()`** and **`DataFrame.compare()`**: Show differences in values between two **`Series`** or **`DataFrame`** objects

## `concat()`

The **`concat()`** function concatenates an arbitrary amount of **`Series`** or **`DataFrame`** objects along an axis while performing optional set logic (union or intersection) of the indexes on the other axes. Like `numpy.concatenate`, **`concat()`** takes a list or dict of homogeneously-typed objects and concatenates them.


> ### How to use this notebook
> Your code is **unchanged** (the same as the pandas docs). The new **"Explained"** cells were added around it.
> For every example:
> 1. **Read** the explanation before the code (what we are trying to do).
> 2. **Read** the code.
> 3. **Run** it, then read the **"What happened"** cell below it and match it with the output.
>
> Tip: try to **guess the output before you run**. If your guess was wrong, reread that part.
>
> **In this part we study:** `concat`, `merge` (+ `validate`, `indicator`, `suffixes`), `join`, `merge_asof`, `compare`.
> **We skip (as the plan says):** joins with two MultiIndexes, `combine_first`, `merge_ordered`.
>
> **The big difference in one line:**
> `concat` puts tables together by **labels** (index / column names) — it does not look at values.
> `merge` / `join` put rows together by **matching values** in a key (like SQL JOIN or Excel VLOOKUP).

In [1]:
import numpy as np
import pandas as pd

#### Explained — basic `concat`

`pd.concat(frames)` takes a **list** of tables and stacks them **on top of each other** (default `axis=0` = add rows).

```
df1  (rows 0-3)      ┐
df2  (rows 4-7)      ├──►  result (rows 0-11)
df3  (rows 8-11)     ┘
```

All three tables have the same columns `A, B, C, D`, so the columns line up perfectly.

In [2]:
df1 = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    },
    index=[0, 1, 2, 3],
)


df2 = pd.DataFrame(
    {
        "A": ["A4", "A5", "A6", "A7"],
        "B": ["B4", "B5", "B6", "B7"],
        "C": ["C4", "C5", "C6", "C7"],
        "D": ["D4", "D5", "D6", "D7"],
    },
    index=[4, 5, 6, 7],
)


df3 = pd.DataFrame(
    {
        "A": ["A8", "A9", "A10", "A11"],
        "B": ["B8", "B9", "B10", "B11"],
        "C": ["C8", "C9", "C10", "C11"],
        "D": ["D8", "D9", "D10", "D11"],
    },
    index=[8, 9, 10, 11],
)


frames = [df1, df2, df3]

result = pd.concat(frames)

result

,A,B,C,D
0,A0,B0,C0,D0
1,A1,B1,C1,D1
2,A2,B2,C2,D2
3,A3,B3,C3,D3
4,A4,B4,C4,D4
5,A5,B5,C5,D5
6,A6,B6,C6,D6
7,A7,B7,C7,D7
8,A8,B8,C8,D8
9,A9,B9,C9,D9


#### What happened

- 4 + 4 + 4 = **12 rows**, same 4 columns.
- The index `0 ... 11` was **kept from the original tables**. It looks clean only because the docs gave each table different index numbers (`0-3`, `4-7`, `8-11`).
- If all three tables had index `0-3`, you would get `0,1,2,3,0,1,2,3,0,1,2,3` — duplicate labels. That is what `ignore_index=True` fixes (see below).

**`concat()`** makes a full copy of the data, and iteratively reusing **`concat()`** can create unnecessary copies. Collect all **`DataFrame`** or **`Series`** objects in a list before using **`concat()`**.

```python
frames = [process_your_file(f) for f in files]
result = pd.concat(frames)

```

**Note**

When concatenating **`DataFrame`** with named axes, pandas will attempt to preserve these index/column names whenever possible. In the case where all inputs share a common name, this name will be assigned to the result. When the input names do not all agree, the result will be unnamed. The same is true for **`MultiIndex`**, but the logic is applied separately on a level-by-level basis.



#### Explained — the two notes above in simple words

**1. Don't call `concat` inside a loop.** Every `concat` makes a **new copy** of all the data. In a loop, the copy gets bigger each time → very slow. Instead, put all tables in a list first, then call `concat` **once**:

```python
frames = [process_your_file(f) for f in files]   # build the list
result = pd.concat(frames)                        # one concat
```

**2. Axis names.** If every input's index has the same *name* (e.g. `df.index.name = "id"`), the result keeps that name. If the names differ, the result's index has no name. You rarely need to think about this.

### Joining logic of the resulting axis

The `join` keyword specifies how to handle axis values that don’t exist in the first **`DataFrame`**.

`join='outer'` takes the union of all axis values.


#### Explained — what "the resulting axis" means

When you concat, there are two directions:

- the direction you **stack along** (`axis=0` → rows, `axis=1` → columns)
- the **other** direction, which must **line up by labels**

`join=` decides what to do with labels on that **other** direction that are **not in every table**:

| `join=` | keeps | like |
|---|---|---|
| `"outer"` (default) | **all** labels from all tables, fills gaps with `NaN` | union ∪ |
| `"inner"` | only labels that are in **every** table | intersection ∩ |

In the next example we use `axis=1` (put tables **side by side**), so the thing that must line up is the **index (row labels)**:

```
df1 index: 0, 1, 2, 3
df4 index:       2, 3, 6, 7
union     : 0, 1, 2, 3, 6, 7     ← outer
intersect :       2, 3           ← inner
```

In [3]:
df4 = pd.DataFrame(
    {
        "B": ["B2", "B3", "B6", "B7"],
        "D": ["D2", "D3", "D6", "D7"],
        "F": ["F2", "F3", "F6", "F7"],
    },
    index=[2, 3, 6, 7],
)


result = pd.concat([df1, df4], axis=1)

result

,A,B,C,D,B,D,F
0,A0,B0,C0,D0,NaN,NaN,NaN
1,A1,B1,C1,D1,NaN,NaN,NaN
2,A2,B2,C2,D2,B2,D2,F2
3,A3,B3,C3,D3,B3,D3,F3
6,NaN,NaN,NaN,NaN,B6,D6,F6
7,NaN,NaN,NaN,NaN,B7,D7,F7


#### What happened (`axis=1`, default `join="outer"`)

- Rows = **union** of the two indexes: `0, 1, 2, 3, 6, 7` (6 rows).
- Rows `0, 1` exist only in `df1` → `df4`'s columns (`B, D, F`) are `NaN`.
- Rows `6, 7` exist only in `df4` → `df1`'s columns (`A, B, C, D`) are `NaN`.
- Rows `2, 3` exist in both → all columns filled.
- Notice **`B` and `D` appear twice**. `concat` does **not** combine columns with the same name — it just places the tables next to each other. (`merge` is the tool that matches things.)

In [4]:
# join = "inner" takes the intersection of the axis value
result = pd.concat([df1, df4], axis = 1, join = 'inner')
result

,A,B,C,D,B,D,F
2,A2,B2,C2,D2,B2,D2,F2
3,A3,B3,C3,D3,B3,D3,F3


#### What happened (`join="inner"`)

Only index labels that are in **both** tables survive: `2` and `3`. No `NaN` anywhere.

In [5]:
# To perform an effective “left” join using the exact index from the original DataFrame, result can be reindexed.
result = pd.concat([df1, df4], axis = 1).reindex(df1.index)
result

,A,B,C,D,B,D,F
0,A0,B0,C0,D0,NaN,NaN,NaN
1,A1,B1,C1,D1,NaN,NaN,NaN
2,A2,B2,C2,D2,B2,D2,F2
3,A3,B3,C3,D3,B3,D3,F3


#### What happened (a "left join" with concat)

`concat` has no `join="left"` option. So the trick is two steps:

1. `pd.concat([df1, df4], axis=1)` → the outer result (rows `0,1,2,3,6,7`)
2. `.reindex(df1.index)` → "keep only rows `0,1,2,3`, in that order"

So rows `6, 7` (only in `df4`) are dropped, and you keep **exactly** `df1`'s rows.

> `reindex(labels)` = give me rows with exactly these labels; drop the others; use `NaN` for labels that don't exist.

### Ignoring indexes on the concatenation axis


#### Explained

Often the index has **no meaning** — it is just row numbers `0, 1, 2...` from `read_csv`.
Then `ignore_index=True` throws the old index away and gives a new one `0 ... n-1`.

In this example we stack **rows** again (default `axis=0`). Watch out:
`df1` has index `0,1,2,3` and `df4` has index `2,3,6,7` → labels `2` and `3` would appear **twice**.

In [6]:
# For DataFrame objects which don’t have a meaningful index, the ignore_index ignores overlapping indexes.
result = pd.concat([df1, df4], ignore_index = True, sort = False)
result

,A,B,C,D,F
0,A0,B0,C0,D0,NaN
1,A1,B1,C1,D1,NaN
2,A2,B2,C2,D2,NaN
3,A3,B3,C3,D3,NaN
4,NaN,B2,NaN,D2,F2
5,NaN,B3,NaN,D3,F3
6,NaN,B6,NaN,D6,F6
7,NaN,B7,NaN,D7,F7


#### What happened

- 4 + 4 = **8 rows**, new index `0 ... 7` (no duplicates).
- Columns = union of both tables: `A, B, C, D, F` (outer is the default).
  - `df1` has no `F` → rows 0-3 have `NaN` in `F`.
  - `df4` has no `A` or `C` → rows 4-7 have `NaN` there.
- `sort=False` → keep the columns in the order they appear, don't sort them A-Z.

**Don't** use `ignore_index=True` when the index **means something** (names, dates, IDs) — you would lose it.

### Concatenating Series and DataFrame together
*(This section is in the docs but was missing from your notebook. The code below is the docs code.)*

#### Explained

You can `concat` a **Series** with a **DataFrame** using `axis=1`. The Series becomes a **new column**.
The Series `name` becomes the column name.

In [7]:
s1 = pd.Series(["X0", "X1", "X2", "X3"], name="X")
result = pd.concat([df1, s1], axis=1)
result

,A,B,C,D,X
0,A0,B0,C0,D0,X0
1,A1,B1,C1,D1,X1
2,A2,B2,C2,D2,X2
3,A3,B3,C3,D3,X3


#### What happened

`s1` has index `0,1,2,3` — same as `df1` — so each value lands on the matching row. The new column is called **`X`** (the Series name).

#### Explained

A Series **without a name** gets a number as its column name. Unnamed Series are numbered `0, 1, 2...` in order.

In [8]:
s2 = pd.Series(["_0", "_1", "_2", "_3"])
result = pd.concat([df1, s2, s2, s2], axis=1)
result

,A,B,C,D,0,1,2
0,A0,B0,C0,D0,_0,_0,_0
1,A1,B1,C1,D1,_1,_1,_1
2,A2,B2,C2,D2,_2,_2,_2
3,A3,B3,C3,D3,_3,_3,_3


#### What happened

The three unnamed copies of `s2` became columns **`0`, `1`, `2`**.

#### Explained

With `ignore_index=True` and `axis=1`, the **column names** are thrown away (because now the concatenation axis is the columns) → columns become `0, 1, 2, ...`.

In [9]:
result = pd.concat([df1, s1], axis=1, ignore_index=True)
result

,0,1,2,3,4
0,A0,B0,C0,D0,X0
1,A1,B1,C1,D1,X1
2,A2,B2,C2,D2,X2
3,A3,B3,C3,D3,X3


#### What happened

Columns `A, B, C, D, X` became `0, 1, 2, 3, 4`. `ignore_index` always acts on the axis you are **concatenating along** — here that is the columns.

### Resulting keys


#### Explained

After stacking, you can't tell which row came from which table. `keys=` adds an **extra outer label** for each input table.
The result gets a **MultiIndex** (two levels of row labels): `(key, old index)`.

```
keys = ['x', 'y', 'z']
         │    │    └── label for rows from df3
         │    └─────── label for rows from df2
         └──────────── label for rows from df1
```

In [10]:
# The keys argument adds another axis level to the resulting index or column (creating a MultiIndex) associate specific keys with each original DataFrame.
result = pd.concat(frames, keys = ['x', 'y', 'z'])
result

A    B    C    D
x 0    A0   B0   C0   D0
  1    A1   B1   C1   D1
  2    A2   B2   C2   D2
  3    A3   B3   C3   D3
y 4    A4   B4   C4   D4
  5    A5   B5   C5   D5
  6    A6   B6   C6   D6
  7    A7   B7   C7   D7
z 8    A8   B8   C8   D8
  9    A9   B9   C9   D9
  10  A10  B10  C10  D10
  11  A11  B11  C11  D11

#### What happened

- The index now has 2 levels: outer = `x / y / z`, inner = the old index `0 ... 11`.
- Rows 0-3 are under `x` (from `df1`), 4-7 under `y` (`df2`), 8-11 under `z` (`df3`).
- Now you can take one table back out: `result.loc["y"]` → only the `df2` rows.
- Very common real use: `pd.concat(monthly_frames, keys=["jan", "feb", "mar"])` then `.reset_index(level=0)` to turn the key into a normal "month" column.

In [11]:
# The keys argument can override the column names when creating a new DataFrame based on existing Series.
s3 = pd.Series([0, 1, 2, 3], name="foo")
s4 = pd.Series([0, 1, 2, 3])
s5 = pd.Series([0, 1, 4, 5])
pd.concat([s3, s4, s5], axis=1)

,foo,0,1
0,0,0,0
1,1,1,1
2,2,2,4
3,3,3,5


#### What happened

With `axis=1`, each Series becomes **one column**. Column names come from the Series `name`:

- `s3` has `name="foo"` → column `foo`
- `s4`, `s5` have no name → numbered `0`, `1`

Ugly names. The next cell fixes them with `keys=`.

In [12]:
pd.concat([s3, s4, s5], axis=1, keys=["red", "blue", "yellow"])

,red,blue,yellow
0,0,0,0
1,1,1,1
2,2,2,4
3,3,3,5


#### What happened

With `axis=1`, `keys=` gives the **column names**: `red, blue, yellow` (in order). It **overrides** `foo`, `0`, `1`.
Same data, better names.

In [13]:
# You can also pass a dict to concat() in which case the dict keys will be used for the keys argument unless other keys argument is specified:
pieces = {'x': df1, 'y': df2, 'z': df3}
result = pd.concat(pieces)
result

A    B    C    D
x 0    A0   B0   C0   D0
  1    A1   B1   C1   D1
  2    A2   B2   C2   D2
  3    A3   B3   C3   D3
y 4    A4   B4   C4   D4
  5    A5   B5   C5   D5
  6    A6   B6   C6   D6
  7    A7   B7   C7   D7
z 8    A8   B8   C8   D8
  9    A9   B9   C9   D9
  10  A10  B10  C10  D10
  11  A11  B11  C11  D11

#### What happened

A **dict** works like a list + keys: the dict keys `'x', 'y', 'z'` are used as the `keys`. The result is exactly the same as the `keys=['x','y','z']` example above. Less typing.

#### Explained

Dict **and** `keys=` together: now `keys` acts like a **selector** — only these dict entries are used, **in this order**.
Guess before you run: which table is missing?

In [14]:
result = pd.concat(pieces, keys = ['z', 'y'])
result

A    B    C    D
z 8    A8   B8   C8   D8
  9    A9   B9   C9   D9
  10  A10  B10  C10  D10
  11  A11  B11  C11  D11
y 4    A4   B4   C4   D4
  5    A5   B5   C5   D5
  6    A6   B6   C6   D6
  7    A7   B7   C7   D7

#### What happened

- `'x'` was not in `keys` → `df1` is **dropped**.
- Order follows `keys`: `z` (`df3`) first, then `y` (`df2`) — not the dict order.

In [15]:
# The MultiIndex created has levels that are constructed from the passed keys and the index of the DataFrame pieces:
result.index.levels

FrozenList([['z', 'y'], [4, 5, 6, 7, 8, 9, 10, 11]])

#### What happened

`.levels` = the list of **unique labels** at each index level:

- level 0: `['z', 'y']` — the keys you used
- level 1: `[4, 5, ..., 11]` — the old index values from `df2` and `df3`

You rarely need `.levels` in practice; it is shown to prepare the next cell.

#### Explained (advanced — rarely needed)

- `levels=[['z', 'y', 'x', 'w']]` → declare the **full list of allowed labels** for the outer level, even labels with no rows (`'w'`).
- `names=["group_key"]` → give the outer index level a **name**. (This one is useful: `reset_index()` will then create a column called `group_key`.)

In [16]:
# levels argument allows specifying resulting levels associated with the keys.
result = pd.concat(
    pieces, keys = ['x', 'y', 'z'], levels = [['z', 'y', 'x', 'w']], names = ["group_key"]
)
result

A    B    C    D
group_key                       
x         0    A0   B0   C0   D0
          1    A1   B1   C1   D1
          2    A2   B2   C2   D2
          3    A3   B3   C3   D3
y         4    A4   B4   C4   D4
          5    A5   B5   C5   D5
          6    A6   B6   C6   D6
          7    A7   B7   C7   D7
z         8    A8   B8   C8   D8
          9    A9   B9   C9   D9
          10  A10  B10  C10  D10
          11  A11  B11  C11  D11

#### What happened

- The rows look the same as before (`x, y, z`).
- The outer level now has the name **`group_key`** (see the header on the left).

In [17]:
result.index.levels

FrozenList([['z', 'y', 'x', 'w'], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]])

#### What happened

Level 0 now contains `'w'` even though no row has `'w'` — because you declared it in `levels=`. That is the only effect of `levels`.

### Appending rows to a DataFrame
*(In the docs, missing from your notebook. Docs code below.)*

#### Explained

There is no `df.append()` any more (removed in pandas 2.0). To add one row:

1. Make the row a **Series** whose index = the column names (`A, B, C, D`).
2. `s2.to_frame()` turns it into a 1-column DataFrame, `.T` (transpose) flips it into a **1-row** DataFrame.
3. `concat` it under `df1`, with `ignore_index=True` for a clean index.

In [18]:
s2 = pd.Series(["X0", "X1", "X2", "X3"], index=["A", "B", "C", "D"])
result = pd.concat([df1, s2.to_frame().T], ignore_index=True)
result

,A,B,C,D
0,A0,B0,C0,D0
1,A1,B1,C1,D1
2,A2,B2,C2,D2
3,A3,B3,C3,D3
4,X0,X1,X2,X3


#### What happened

`df1` had 4 rows → now **5 rows**. The new row `X0 X1 X2 X3` is at index `4`.
Values went to the right columns because the Series **index** matched the **column names**.

---
### `concat` summary

```python
pd.concat([a, b, c])                          # stack rows
pd.concat([a, b, c], ignore_index=True)       # stack rows + fresh 0..n-1 index
pd.concat([a, b], join="inner")               # keep only labels shared by all
pd.concat([a, b], axis=1)                     # side by side, lined up by index
pd.concat([a, b, c], keys=["x", "y", "z"])    # remember which table each row came from
```

## `merge()`

**`merge()`** performs join operations similar to relational databases like SQL. Users who are familiar with SQL but new to pandas can reference a comparison with SQL.

### Merge types

**`merge()`** implements common SQL style joining operations.

* **one-to-one**: joining two **`DataFrame`** objects on their indexes which must contain unique values.
* **many-to-one**: joining a unique index to one or more columns in a different **`DataFrame`**.
* **many-to-many**: joining columns on columns.

**Note**

When joining columns on columns, potentially a many-to-many join, any indexes on the passed **`DataFrame`** objects **will be discarded**.

For a **many-to-many** join, if a key combination appears more than once in both tables, the **`DataFrame`** will have the **Cartesian product** of the associated data.


#### Explained — the 3 relationships in simple words

`merge` matches rows where the **key value is equal** in both tables. What matters is **how many times each key appears on each side**:

| Relationship | Left key appears | Right key appears | Example |
|---|---|---|---|
| **one-to-one** | once | once | person ↔ passport |
| **many-to-one** | many times | once | many orders ↔ one customer |
| **many-to-many** | many times | many times | usually a **mistake** — rows multiply |

**Cartesian product** = every copy on the left pairs with every copy on the right.
If key `K1` appears **2 times** on the left and **3 times** on the right → **2 × 3 = 6 rows** for `K1`.

**"indexes will be discarded"** = when you merge on columns, the result gets a fresh `0, 1, 2...` index. The old row labels are not kept.

#### Explained — simplest merge

`on="key"` = match rows where the column `key` has the same value in both tables. Default `how="inner"`.

Every key `K0 ... K3` appears **once** in each table → this is a **one-to-one** merge.

> Small typo in the first line: `left = pd.left = pd.DataFrame(...)`. It still works, but it also creates a strange
> attribute `pd.left` on the pandas module. The docs have just `left = pd.DataFrame(...)`. (Code left as you wrote it.)

In [19]:
left = pd.left = pd.DataFrame(
    {
        "key": ["K0", "K1", "K2", "K3"],
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
    }
)

right = pd.DataFrame(
    {
        "key": ["K0", "K1", "K2", "K3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    }
)

result = pd.merge(left, right, on="key")
result

,key,A,B,C,D
0,K0,A0,B0,C0,D0
1,K1,A1,B1,C1,D1
2,K2,A2,B2,C2,D2
3,K3,A3,B3,C3,D3


#### What happened

- `key` appears **once** in the result (it is the join column, so pandas keeps one copy).
- Columns `A, B` came from `left`, `C, D` from `right`, placed on the row with the **same key**.
- 4 keys matched → **4 rows**.

The `how` argument to **`merge()`** specifies which keys are included in the resulting table. If a key combination **does not appear** in either the left or right tables, the values in the joined table will be `NA`. Here is a summary of the `how` options and their SQL equivalent names:

| Merge method | SQL Join Name      | Description                                         |
| ------------ | ------------------ | --------------------------------------------------- |
| `left`       | `LEFT OUTER JOIN`  | Use keys from left frame only                       |
| `right`      | `RIGHT OUTER JOIN` | Use keys from right frame only                      |
| `outer`      | `FULL OUTER JOIN`  | Use union of keys from both frames                  |
| `inner`      | `INNER JOIN`       | Use intersection of keys from both frames           |
| `cross`      | `CROSS JOIN`       | Create the cartesian product of rows of both frames |


#### Explained — the next 5 examples

The next tables use **two key columns** together: `on=["key1", "key2"]`.
A row matches only if **both** `key1` **and** `key2` are equal. Think of each pair as one key:

```
left keys             right keys
(K0, K0)  A0 B0       (K0, K0)  C0 D0     ← match
(K0, K1)  A1 B1       (K1, K0)  C1 D1     ─┐ (K1,K0) appears TWICE on the right
(K1, K0)  A2 B2       (K1, K0)  C2 D2     ─┘
(K2, K1)  A3 B3       (K2, K0)  C3 D3
```

Matches:

| key pair | in left? | in right? |
|---|---|---|
| (K0, K0) | yes | yes (1×) |
| (K0, K1) | yes | **no** |
| (K1, K0) | yes | yes (**2×**) |
| (K2, K1) | yes | **no** |
| (K2, K0) | **no** | yes |

Keep this table open. Every result below comes from it.

In [20]:
left = pd.DataFrame(
   {
      "key1": ["K0", "K0", "K1", "K2"],
      "key2": ["K0", "K1", "K0", "K1"],
      "A": ["A0", "A1", "A2", "A3"],
      "B": ["B0", "B1", "B2", "B3"],
   }
)

right = pd.DataFrame(
   {
      "key1": ["K0", "K1", "K1", "K2"],
      "key2": ["K0", "K0", "K0", "K0"],
      "C": ["C0", "C1", "C2", "C3"],
      "D": ["D0", "D1", "D2", "D3"],
   }
)

result = pd.merge(left, right, how="left", on=["key1", "key2"])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K0,K1,A1,B1,NaN,NaN
2,K1,K0,A2,B2,C1,D1
3,K1,K0,A2,B2,C2,D2
4,K2,K1,A3,B3,NaN,NaN


#### What happened — `how="left"` → 5 rows

Keep **every left key pair**, add right columns where they match:

- (K0, K0) → matched → `C0 D0`
- (K0, K1) → no match → `NaN NaN`
- (K1, K0) → matches **2 right rows** → the left row `A2 B2` is **repeated twice** (once with `C1 D1`, once with `C2 D2`)
- (K2, K1) → no match → `NaN NaN`

⚠️ **Left had 4 rows, result has 5.** A left merge keeps every left row *at least* once, but can **add** rows when the right key repeats. This is why `validate=` exists (below).

In [21]:
result = pd.merge(left, right, how="right", on=["key1", "key2"])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K1,K0,A2,B2,C1,D1
2,K1,K0,A2,B2,C2,D2
3,K2,K0,NaN,NaN,C3,D3


#### What happened — `how="right"` → 4 rows

Keep **every right row**:

- (K0, K0) → `A0 B0`
- (K1, K0) → `A2 B2` (twice, once for each right row `C1` and `C2`)
- (K2, K0) → not in left → `A`, `B` are `NaN`

(K0, K1) and (K2, K1) are gone — they are only in the left.

In [22]:
result = pd.merge(left, right, how="outer", on=["key1", "key2"])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K0,K1,A1,B1,NaN,NaN
2,K1,K0,A2,B2,C1,D1
3,K1,K0,A2,B2,C2,D2
4,K2,K0,NaN,NaN,C3,D3
5,K2,K1,A3,B3,NaN,NaN


#### What happened — `how="outer"` → 6 rows

Keep **everything** from both sides: the 3 matched rows (K0,K0), (K1,K0) ×2 + left-only (K0,K1), (K2,K1) + right-only (K2,K0).
Missing sides are `NaN`.

In [23]:
result = pd.merge(left, right, how="inner", on=["key1", "key2"])
result

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,C0,D0
1,K1,K0,A2,B2,C1,D1
2,K1,K0,A2,B2,C2,D2


#### What happened — `how="inner"` → 3 rows

Only key pairs found in **both**: (K0,K0) once and (K1,K0) twice. Everything with `NaN` from the outer result is gone.

**Row count check:** inner 3 ≤ left 5, right 4 ≤ outer 6. Inner is always the smallest, outer the biggest.

#### Explained — `how="cross"`

No key at all. Every left row is paired with **every** right row. Rows = `len(left) × len(right)`.
Useful for building "all combinations" (e.g. every size × every color).

In [24]:
result = pd.merge(left, right, how="cross")
result

,key1_x,key2_x,A,B,key1_y,key2_y,C,D
0,K0,K0,A0,B0,K0,K0,C0,D0
1,K0,K0,A0,B0,K1,K0,C1,D1
2,K0,K0,A0,B0,K1,K0,C2,D2
3,K0,K0,A0,B0,K2,K0,C3,D3
4,K0,K1,A1,B1,K0,K0,C0,D0
5,K0,K1,A1,B1,K1,K0,C1,D1
6,K0,K1,A1,B1,K1,K0,C2,D2
7,K0,K1,A1,B1,K2,K0,C3,D3
8,K1,K0,A2,B2,K0,K0,C0,D0
9,K1,K0,A2,B2,K1,K0,C1,D1


#### What happened

- 4 × 4 = **16 rows**. Row 0 of left pairs with all 4 right rows, then row 1, and so on.
- `key1`, `key2` exist in both tables but were **not** used as keys → pandas renamed them `key1_x` (left) and `key1_y` (right). More on `_x`/`_y` in *Overlapping value columns*.
- ⚠️ Grows fast: 10,000 × 10,000 = 100 million rows. Only use on small tables.

#### Explained — merging a Series that has a MultiIndex

`merge` matches on **columns**. If your Series keeps its keys in the **index** (here: `Let` and `Num`), use
`.reset_index()` to turn those index levels into normal columns, then merge.
The next 3 cells: make `df`, make `ser`, then merge.

In [25]:
# You can merge Series and a DataFrame with a MultiIndex if the names of the MultiIndex correspond to the columns from the DataFrame. You can also transform the Series to a DataFrame using Series.reset_index() before merging:
df = pd.DataFrame({'Let': ['A', 'B', 'C'], 'Num': [1,2,3]})
df

,Let,Num
0,A,1
1,B,2
2,C,3


In [26]:
ser = pd.Series(
    ['a', 'b', 'c', 'd', 'e', 'f'],
    index = pd.MultiIndex.from_arrays(
        [['A', 'B', 'C'] * 2, [1, 2, 3, 4, 5, 6]], names = ['Let', 'Num']
    ),
)
ser

Let  Num
A    1      a
B    2      b
C    3      c
A    4      d
B    5      e
C    6      f
dtype: str

#### What happened

`ser` has a 2-level index `(Let, Num)` and values `a ... f`. Try `ser.reset_index()` in a scratch cell — you'll see columns `Let`, `Num`, and `0`.
The values column is called **`0`** because the Series has no `name`.

In [27]:
pd.merge(df, ser.reset_index(), on = ['Let', 'Num'])

,Let,Num,0
0,A,1,a
1,B,2,b
2,C,3,c


#### What happened

Match on **both** `Let` and `Num` (inner merge):

- (A,1), (B,2), (C,3) are in both → kept, with values `a, b, c`.
- (A,4), (B,5), (C,6) are only in `ser` → dropped.

#### Explained — duplicate keys on BOTH sides (many-to-many)

Every row has `B = 2`. Left has it **2 times**, right has it **3 times**. Guess the row count before you run.

In [28]:
# Performing an outer join with duplicate join keys in DataFrame:
left = pd.DataFrame({"A": [1, 2], "B": [2, 2]})

right = pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]})

result = pd.merge(left, right, on="B", how="outer")

result

,A_x,B,A_y
0,1,2,4
1,1,2,5
2,1,2,6
3,2,2,4
4,2,2,5
5,2,2,6


#### What happened

- **2 × 3 = 6 rows** — the Cartesian product. Each left row (A=1, A=2) is paired with each right row (A=4, 5, 6).
- Both tables have a column `A` that is **not** the key → it became `A_x` (left) and `A_y` (right).
- On a real dataset this is how 10,000 rows silently become millions. Next section: how to catch it.

### Merge key uniqueness
The `validate` argument checks whether the uniqueness of merge keys. Key uniqueness is checked before merge operations and can protect against memory overflows and unexpected key duplication.

#### Explained — ⭐ `validate=` (roadmap rule: "validate every merge")

**The problem:** you *think* a key is unique (one row per customer), but the data has a duplicate.
`merge` does **not** complain — it silently duplicates rows, and your totals become wrong.

**The fix:** tell pandas what you expect. It checks the keys **before** merging. If you're wrong → `MergeError`, and nothing is merged.

| `validate=` | short | checks that… |
|---|---|---|
| `"one_to_one"` | `"1:1"` | key is unique in **left** AND in **right** |
| `"one_to_many"` | `"1:m"` | key is unique in **left** (right may repeat) |
| `"many_to_one"` | `"m:1"` | key is unique in **right** (left may repeat) |
| `"many_to_many"` | `"m:m"` | nothing (just documents what you expect) |

**How to choose:** read it as *left*\_to\_*right*. Ask "for one key, how many rows in the left? in the right?"
Most common in real work: `orders.merge(customers, validate="many_to_one")` — many orders, one customer each.

**The next cell is supposed to fail.** An error here is the lesson, not a mistake.

In [29]:
left = pd.DataFrame({"A": [1, 2], "B": [1, 2]})
right = pd.DataFrame({"A": [4, 5, 6], "B": [2, 2, 2]})
result = pd.merge(left, right, on="B", how="outer", validate="one_to_one")

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  B
 2
 2 ...

#### What happened

- `left["B"]` = `[1, 2]` → unique ✅
- `right["B"]` = `[2, 2, 2]` → **not** unique ❌
- You promised `one_to_one` → both must be unique → pandas raised:
  **`MergeError: Merge keys are not unique in right dataset; not a one-to-one merge`**

How to read a long traceback: **skip to the last line**. It names the table (left/right) and the rule that broke.
(Newer pandas also lists the duplicate keys under it.)

Next step on real data would be to look at the duplicates: `right[right.duplicated("B", keep=False)]`.

If the user is aware of the duplicates in the right DataFrame but wants to ensure there are no duplicates in the left DataFrame, one can use the validate='one_to_many' argument instead, which will not raise an exception.

In [30]:
pd.merge(left, right, on ="B", how = "outer", validate = "one_to_many")

,A_x,B,A_y
0,1,1,NaN
1,2,2,4.0
2,2,2,5.0
3,2,2,6.0


#### What happened

- `one_to_many` only checks that the **left** is unique → `[1, 2]` is unique → OK, no error.
- Result (outer): **4 rows**
  - `B=1` → only in left → `A_y` is `NaN`
  - `B=2` → 1 left row × 3 right rows → **3 rows** (`A_y` = 4, 5, 6)
- `A_y` shows `4.0` instead of `4` — a column containing `NaN` becomes float.

**Takeaway:** the duplicates are still there. `validate` doesn't remove anything — it just confirms they are **expected**.

### Merge result indicator

**`merge()`** accepts the argument `indicator`. If `True`, a Categorical-type column called `_merge` will be added to the output object that takes on values:

| Observation Origin                | `_merge` value |
| --------------------------------- | -------------- |
| Merge key only in `'left'` frame  | `left_only`    |
| Merge key only in `'right'` frame | `right_only`   |
| Merge key in both frames          | `both`         |


#### Explained — ⭐ `indicator=True`

Adds a column **`_merge`** that tells you, for each row, **where its key was found**.
Most useful with `how="outer"`, because only outer can show all three values.

⚠️ The next cell **re-uses the names `df1` and `df2`** for new small tables. The old `df1`/`df2` from the concat section are overwritten.

In [31]:
df1 = pd.DataFrame({'col1': [0,1], 'col_left': ['a', 'b']})
df2 = pd.DataFrame({'col1': [1, 2, 2], 'col_right': [2, 2, 2]})
pd.merge(df1, df2, on = 'col1', how = 'outer', indicator = True)

,col1,col_left,col_right,_merge
0,0,a,NaN,left_only
1,1,b,2.0,both
2,2,NaN,2.0,right_only
3,2,NaN,2.0,right_only


#### What happened

```
df1 col1: 0, 1          df2 col1: 1, 2, 2
```

- `col1 = 0` → only in `df1` → **`left_only`** (`col_right` is `NaN`)
- `col1 = 1` → in both → **`both`**
- `col1 = 2` → only in `df2`, and it appears **twice** there → 2 rows, both **`right_only`** (`col_left` is `NaN`)

**How you use it on real data** — count the matches:

```python
result["_merge"].value_counts()
# both          1
# right_only    2
# left_only     1
```

- Big `left_only` / `right_only` → keys didn't match. Check spaces, upper/lower case, int vs string (`1` vs `"1"`).
- Find the unmatched rows: `result[result["_merge"] == "left_only"]`
- In a `how="left"` merge you will never see `right_only`; in `how="inner"` only `both`.
- Drop it afterwards: `result = result.drop(columns="_merge")` (it would clash with the next merge's `_merge`).

In [32]:
# A string argument to indicator will use the value as the name for the indicator column.
pd.merge(df1, df2, on = 'col1', how = 'outer', indicator = 'indicator_column')

,col1,col_left,col_right,indicator_column
0,0,a,NaN,left_only
1,1,b,2.0,both
2,2,NaN,2.0,right_only
3,2,NaN,2.0,right_only


#### What happened

Same result, but the column is called **`indicator_column`** instead of `_merge`. Useful when you do several merges in a row.

### Overlapping value columns
The merge `suffixes` argument takes a tuple or list of strings to append to overlapping column names in the input **`DataFrame`** to disambiguate the result columns:

#### Explained

Both tables have a column **`v`**, and `v` is **not** the key. A table can't have two columns both called `v`,
so pandas adds endings: **`_x`** = from the left, **`_y`** = from the right.

In [33]:
left = pd.DataFrame({"k": ["K0", "K1", "K2"], "v": [1, 2, 3]})

right = pd.DataFrame({"k": ["K0", "K0", "K3"], "v": [4, 5, 6]})

result = pd.merge(left, right, on="k")

result

,k,v_x,v_y
0,K0,1,4
1,K0,1,5


#### What happened

- Default `how="inner"`: only `K0` is in both → the others are gone.
- `K0` is **once** in left and **twice** in right → **2 rows**.
- `v_x` = left's `v` (1), `v_y` = right's `v` (4 and 5).

In [34]:
result = pd.merge(left, right, on="k", suffixes=("_l", "_r"))

result

,k,v_l,v_r
0,K0,1,4
1,K0,1,5


#### What happened

Same result, but the endings are the ones you chose: `v_l`, `v_r`. On real data use meaningful names, e.g. `suffixes=("_2024", "_2025")`.

**Tip:** if you see `_x` / `_y` in a result and the shared column is really part of the key, you probably forgot it in `on=`.
Put it there and there will be only one column.

## `DataFrame.join()`

**`DataFrame.join()`** combines the columns of multiple, potentially differently-indexed **`DataFrame`** into a single result **`DataFrame`**.


#### Explained — `join` is a shortcut for "merge on the index"

| | `merge` | `join` |
|---|---|---|
| matches on | columns (by default) | the **index** of the right table |
| default `how` | `"inner"` | **`"left"`** |
| written as | `pd.merge(a, b, on=...)` | `a.join(b)` |

In the next example the keys `K0, K1...` are the **index** (row labels), not a column.

```
left index : K0, K1, K2
right index: K0,     K2, K3
```

In [35]:
left = pd.DataFrame(
    {"A": ["A0", "A1", "A2"], "B": ["B0", "B1", "B2"]}, index=["K0", "K1", "K2"]
)

right = pd.DataFrame(
    {"C": ["C0", "C2", "C3"], "D": ["D0", "D2", "D3"]}, index=["K0", "K2", "K3"]
)

result = left.join(right)
result

,A,B,C,D
K0,A0,B0,C0,D0
K1,A1,B1,NaN,NaN
K2,A2,B2,C2,D2


#### What happened (default `how="left"`)

- Keep all of `left`'s index: `K0, K1, K2`.
- `K1` is not in `right` → `C`, `D` are `NaN`.
- `K3` is only in `right` → not shown.

In [36]:
result = left.join(right, how="outer")
result

,A,B,C,D
K0,A0,B0,C0,D0
K1,A1,B1,NaN,NaN
K2,A2,B2,C2,D2
K3,NaN,NaN,C3,D3


#### What happened (`how="outer"`)

Union of both indexes: `K0, K1, K2, K3`. `K1` has `NaN` in `C, D`; `K3` has `NaN` in `A, B`.

In [37]:
result = left.join(right, how="inner")
result

,A,B,C,D
K0,A0,B0,C0,D0
K2,A2,B2,C2,D2


#### What happened (`how="inner"`)

Only labels in **both**: `K0` and `K2`.

#### Explained — `join(on=...)`: a **column** on the left, the **index** on the right

This is the classic **lookup table** pattern (like VLOOKUP):

- `left` has a normal column `key` with values `K0, K1, K0, K1` (repeated).
- `right` uses `K0, K1` as its **index**.
- `left.join(right, on="key")` → for each left row, look up its `key` in `right`'s index and bring `C`, `D` over.

In [38]:
# DataFrame.join() takes an optional on argument which may be a column or multiple column names that the passed DataFrame is to be aligned.
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "key": ["K0", "K1", "K0", "K1"],
    }
)

right = pd.DataFrame({"C": ["C0", "C1"], "D": ["D0", "D1"]}, index=["K0", "K1"])
result = left.join(right, on="key")
result

,A,B,key,C,D
0,A0,B0,K0,C0,D0
1,A1,B1,K1,C1,D1
2,A2,B2,K0,C0,D0
3,A3,B3,K1,C1,D1


#### What happened

- All 4 left rows kept, in the **same order** (default `how="left"`).
- Rows with `key = K0` got `C0, D0`; rows with `K1` got `C1, D1`.
- The right table's `K0` row was **re-used** for 2 left rows — this is **many-to-one**, which is normal for a lookup table.
  (You can check it: `left.join(right, on="key", validate="many_to_one")` — works in pandas ≥ 1.5.)

#### Explained — the same thing written with `merge`

- `left_on="key"` → on the left, use the **column** `key`
- `right_index=True` → on the right, use the **index**
- `how="left"` → keep all left rows (this is what `join` does by default)
- `sort=False` → keep the original row order

In [39]:
result = pd.merge(
    left, right, left_on="key", right_index=True, how="left", sort=False
)
result

,A,B,key,C,D
0,A0,B0,K0,C0,D0
1,A1,B1,K1,C1,D1
2,A2,B2,K0,C0,D0
3,A3,B3,K1,C1,D1


#### What happened

Exactly the same table as the `join` above. `join` is just shorter to write.
Use whichever reads more clearly — `merge` is more flexible, `join` is quicker for index lookups.

In [40]:
# To join on multiple keys, the passed DataFrame must have a MultiIndex:

#### Explained — join on multiple keys
*(Docs code below — the continuation of your last cell.)*

To look up **two** columns at once (`key1`, `key2`), the right table needs a **2-level index** (MultiIndex).
The pairs in the left's columns are matched to the pairs in the right's index.

```
left (key1, key2)       right index
(K0, K0)  A0 B0         (K0, K0)  C0 D0   ← match
(K0, K1)  A1 B1         (K1, K0)  C1 D1   ← match for (K1,K0)
(K1, K0)  A2 B2         (K2, K0)  C2 D2   ← no left row has (K2,K0)
(K2, K1)  A3 B3         (K2, K1)  C3 D3   ← match for (K2,K1)
```

In [41]:
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "key1": ["K0", "K0", "K1", "K2"],
        "key2": ["K0", "K1", "K0", "K1"],
    }
)

index = pd.MultiIndex.from_tuples(
    [("K0", "K0"), ("K1", "K0"), ("K2", "K0"), ("K2", "K1")]
)
right = pd.DataFrame(
    {"C": ["C0", "C1", "C2", "C3"], "D": ["D0", "D1", "D2", "D3"]}, index=index
)
result = left.join(right, on=["key1", "key2"])
result

,A,B,key1,key2,C,D
0,A0,B0,K0,K0,C0,D0
1,A1,B1,K0,K1,NaN,NaN
2,A2,B2,K1,K0,C1,D1
3,A3,B3,K2,K1,C3,D3


#### What happened (default `how="left"`)

- (K0,K0) → `C0 D0`, (K1,K0) → `C1 D1`, (K2,K1) → `C3 D3`
- (K0,K1) → not in right → `NaN`
- Right's (K2,K0) (`C2 D2`) is not used — no left row has that pair.

In [42]:
result = left.join(right, on=["key1", "key2"], how="inner")
result

,A,B,key1,key2,C,D
0,A0,B0,K0,K0,C0,D0
2,A2,B2,K1,K0,C1,D1
3,A3,B3,K2,K1,C3,D3


#### What happened (`how="inner"`)

Row 1 (K0,K1) is dropped because it had no match. Notice the index is `0, 2, 3` — `join` **keeps the left's index** (unlike `merge`, which resets it).

**Also good to know** (from the docs' join section): if both tables have a column with the same name, `join` does **not**
add `_x`/`_y` automatically — it raises an error. You must give `lsuffix="_l", rsuffix="_r"`.

**Skipped per plan:** joining a single Index to a MultiIndex, joining with two MultiIndexes, `combine_first`, `merge_ordered`.

## `merge_asof()`
*(Docs code below.)*

`merge_asof()` performs an ordered left-join, similar to a left merge, except that it matches on the **nearest key** rather than an equal key.
For each row in the left `DataFrame`, the **last** row in the right `DataFrame` whose `on` key is **less than or equal** to the left's key is selected. Both `DataFrame` must be **sorted by the key**.

Optionally it can match within a group (`by=`), and limit how far away a match can be (`tolerance=`).

#### Explained

**Real-life problem:** a stock is traded at some time. The market price (quote: `bid`/`ask`) changes at other times.
For each trade, you want **the quote that was valid at that moment** = the **last quote at or before** the trade time.
An exact `merge` on time fails, because the times almost never match exactly.

Rules to remember:

1. Like a **left** merge: every left row (trade) is kept, and gets **at most one** match.
2. Default = look **backward**: last right row with time **≤** left time.
3. Both tables must be **sorted** by the `on` column.
4. `by="ticker"` → only match a MSFT trade with a MSFT quote (exact match on ticker first, then nearest time).

In [43]:
trades = pd.DataFrame(
    {
        "time": pd.to_datetime(
            [
                "20160525 13:30:00.023",
                "20160525 13:30:00.038",
                "20160525 13:30:00.048",
                "20160525 13:30:00.048",
                "20160525 13:30:00.048",
            ]
        ),
        "ticker": ["MSFT", "MSFT", "GOOG", "GOOG", "AAPL"],
        "price": [51.95, 51.95, 720.77, 720.92, 98.00],
        "quantity": [75, 155, 100, 100, 100],
    },
    columns=["time", "ticker", "price", "quantity"],
)

quotes = pd.DataFrame(
    {
        "time": pd.to_datetime(
            [
                "20160525 13:30:00.023",
                "20160525 13:30:00.023",
                "20160525 13:30:00.030",
                "20160525 13:30:00.041",
                "20160525 13:30:00.048",
                "20160525 13:30:00.049",
                "20160525 13:30:00.072",
                "20160525 13:30:00.075",
            ]
        ),
        "ticker": ["GOOG", "MSFT", "MSFT", "MSFT", "GOOG", "AAPL", "GOOG", "MSFT"],
        "bid": [720.50, 51.95, 51.97, 51.99, 720.50, 97.99, 720.50, 52.01],
        "ask": [720.93, 51.96, 51.98, 52.00, 720.93, 98.01, 720.88, 52.03],
    },
    columns=["time", "ticker", "bid", "ask"],
)
trades

,time,ticker,price,quantity
0,2016-05-25 13:30:00.023,MSFT,51.95,75
1,2016-05-25 13:30:00.038,MSFT,51.95,155
2,2016-05-25 13:30:00.048,GOOG,720.77,100
3,2016-05-25 13:30:00.048,GOOG,720.92,100
4,2016-05-25 13:30:00.048,AAPL,98.00,100


In [44]:
quotes

,time,ticker,bid,ask
0,2016-05-25 13:30:00.023,GOOG,720.50,720.93
1,2016-05-25 13:30:00.023,MSFT,51.95,51.96
2,2016-05-25 13:30:00.030,MSFT,51.97,51.98
3,2016-05-25 13:30:00.041,MSFT,51.99,52.00
4,2016-05-25 13:30:00.048,GOOG,720.50,720.93
5,2016-05-25 13:30:00.049,AAPL,97.99,98.01
6,2016-05-25 13:30:00.072,GOOG,720.50,720.88
7,2016-05-25 13:30:00.075,MSFT,52.01,52.03


#### Explained — read the times as milliseconds

All times are `13:30:00.xxx`, so only the last 3 digits matter. Per ticker:

```
MSFT quotes at: .023  .030  .041  .075        MSFT trades at: .023  .038
GOOG quotes at: .023  .048  .072              GOOG trades at: .048  .048
AAPL quotes at: .049                          AAPL trades at: .048
```

Guess before you run: which quote does each trade get?

In [45]:
result = pd.merge_asof(trades, quotes, on="time", by="ticker")
result

,time,ticker,price,quantity,bid,ask
0,2016-05-25 13:30:00.023,MSFT,51.95,75,51.95,51.96
1,2016-05-25 13:30:00.038,MSFT,51.95,155,51.97,51.98
2,2016-05-25 13:30:00.048,GOOG,720.77,100,720.50,720.93
3,2016-05-25 13:30:00.048,GOOG,720.92,100,720.50,720.93
4,2016-05-25 13:30:00.048,AAPL,98.00,100,NaN,NaN


#### What happened

| trade | last same-ticker quote at or before it | result |
|---|---|---|
| MSFT .023 | MSFT .023 (exact time counts, because ≤) | 51.95 / 51.96 |
| MSFT .038 | MSFT .030 (.041 is later, not allowed) | 51.97 / 51.98 |
| GOOG .048 (×2) | GOOG .048 (exact) | 720.50 / 720.93 |
| AAPL .048 | the only AAPL quote is .049 → **later** → none | `NaN` |

The GOOG quote at `.023` is not used for MSFT trades — `by="ticker"` keeps tickers separate.

#### Explained — `tolerance=`

`tolerance=pd.Timedelta("2ms")` → a quote counts only if it is **at most 2 ms** before the trade. Older quotes are "too stale" → `NaN`.

In [46]:
result = pd.merge_asof(trades, quotes, on="time", by="ticker", tolerance=pd.Timedelta("2ms"))
result

,time,ticker,price,quantity,bid,ask
0,2016-05-25 13:30:00.023,MSFT,51.95,75,51.95,51.96
1,2016-05-25 13:30:00.038,MSFT,51.95,155,NaN,NaN
2,2016-05-25 13:30:00.048,GOOG,720.77,100,720.50,720.93
3,2016-05-25 13:30:00.048,GOOG,720.92,100,720.50,720.93
4,2016-05-25 13:30:00.048,AAPL,98.00,100,NaN,NaN


#### What happened

- MSFT .023 → quote .023 is 0 ms away → kept.
- MSFT .038 → nearest earlier is .030 = **8 ms** away > 2 ms → now **`NaN`**.
- GOOG .048 → quote .048 is 0 ms away → kept.
- AAPL → still `NaN`.

#### Explained — `allow_exact_matches=False`

- `tolerance` is now **10 ms**.
- `allow_exact_matches=False` → a quote at the **exact same time** does **not** count. Only **strictly earlier** (`<` instead of `≤`).
  (Idea: a quote at the same millisecond may have been caused by the trade itself.)

In [47]:
result = pd.merge_asof(
    trades,
    quotes,
    on="time",
    by="ticker",
    tolerance=pd.Timedelta("10ms"),
    allow_exact_matches=False,
)
result

,time,ticker,price,quantity,bid,ask
0,2016-05-25 13:30:00.023,MSFT,51.95,75,NaN,NaN
1,2016-05-25 13:30:00.038,MSFT,51.95,155,51.97,51.98
2,2016-05-25 13:30:00.048,GOOG,720.77,100,NaN,NaN
3,2016-05-25 13:30:00.048,GOOG,720.92,100,NaN,NaN
4,2016-05-25 13:30:00.048,AAPL,98.00,100,NaN,NaN


#### What happened

- MSFT .023 → the exact .023 quote is not allowed, nothing earlier → `NaN`.
- MSFT .038 → .030 is 8 ms before (≤ 10 ms) → **51.97 / 51.98**. The only row with a match.
- GOOG .048 → exact .048 not allowed; next earlier is .023 = 25 ms > 10 ms → `NaN`.
- AAPL → `NaN`.

**Other options you may meet:** `direction="forward"` (next row after), `direction="nearest"` (closest either way).
If you forget to sort: `ValueError: left keys must be sorted` → fix with `df.sort_values("time")`.

## `compare()`
*(Docs code below.)*

The `Series.compare()` and `DataFrame.compare()` methods allow you to compare two `DataFrame` or `Series`, respectively, and summarize their differences.

#### Explained

Use it to check your **cleaning**: compare the table **before** and **after**, and see only the cells that changed.
Both tables must have the **same shape, index and columns** (it compares cell by cell). So compare **before** you drop rows or add columns.

In the docs example, `df2` is a copy of `df` with **2 cells changed**:

- row 0, `col1`: `"a"` → `"c"`
- row 2, `col3`: `3.0` → `4.0`

In [48]:
df = pd.DataFrame(
    {
        "col1": ["a", "a", "b", "b", "a"],
        "col2": [1.0, 2.0, 3.0, np.nan, 5.0],
        "col3": [1.0, 2.0, 3.0, 4.0, 5.0],
    },
    columns=["col1", "col2", "col3"],
)
df

,col1,col2,col3
0,a,1.0,1.0
1,a,2.0,2.0
2,b,3.0,3.0
3,b,NaN,4.0
4,a,5.0,5.0


In [49]:
df2 = df.copy()
df2.loc[0, "col1"] = "c"
df2.loc[2, "col3"] = 4.0
df2

,col1,col2,col3
0,c,1.0,1.0
1,a,2.0,2.0
2,b,3.0,4.0
3,b,NaN,4.0
4,a,5.0,5.0


In [50]:
df.compare(df2)

col1       col3      
  self other self other
0    a     c  NaN   NaN
2  NaN   NaN  3.0   4.0

#### What happened

- Only rows **0 and 2** appear (the rows with a change), and only columns **`col1`, `col3`** (the columns with a change). `col2` is not shown — nothing changed there.
- Each column has two sub-columns: **`self`** = `df` (the one you called `.compare` on, "before"), **`other`** = `df2` ("after").
- `NaN` here means **"no difference in this cell"**, not missing data.
- Row 3, `col2` is `NaN` in **both** tables → counts as **equal** → not shown.

Tip (pandas ≥ 1.5): `df.compare(df2, result_names=("before", "after"))` gives clearer labels than `self` / `other`.

#### Explained — `align_axis=0`

Put `self` and `other` on **separate rows** instead of side-by-side columns. Easier to read when the table has many columns.

In [51]:
df.compare(df2, align_axis=0)

col1  col3
0 self     a   NaN
  other    c   NaN
2 self   NaN   3.0
  other  NaN   4.0

#### What happened

Each changed row now appears twice: `(0, self)` then `(0, other)`, `(2, self)` then `(2, other)`. Read them top to bottom as before → after.

#### Explained — `keep_shape=True`

Keep **all** rows and columns, even where nothing changed. Unchanged cells still show `NaN`.

In [52]:
df.compare(df2, keep_shape=True)

col1       col2       col3      
  self other self other self other
0    a     c  NaN   NaN  NaN   NaN
1  NaN   NaN  NaN   NaN  NaN   NaN
2  NaN   NaN  NaN   NaN  3.0   4.0
3  NaN   NaN  NaN   NaN  NaN   NaN
4  NaN   NaN  NaN   NaN  NaN   NaN

#### What happened

All 5 rows and all 3 columns are shown. Only row 0 `col1` and row 2 `col3` have values; everything else is `NaN` = no change.

#### Explained — `keep_equal=True`

Together with `keep_shape=True`: show the **real values** for unchanged cells instead of `NaN`. Now you see the whole table before and after.

In [53]:
df.compare(df2, keep_shape=True, keep_equal=True)

col1       col2       col3      
  self other self other self other
0    a     c  1.0   1.0  1.0   1.0
1    a     a  2.0   2.0  2.0   2.0
2    b     b  3.0   3.0  3.0   4.0
3    b     b  NaN   NaN  4.0   4.0
4    a     a  5.0   5.0  5.0   5.0

#### What happened

Every cell shows its real value. Changed cells are the ones where `self ≠ other`: row 0 `a → c`, row 2 `3.0 → 4.0`.
Row 3 `col2` shows `NaN NaN` — that is a real missing value in both tables.

**Cleaning check you can use:** count how many cells changed in each column:

```python
df.compare(df2, keep_shape=True).xs("self", axis=1, level=1).notna().sum()
```

---
## ⭐ On your dataset (task from the plan)

For **every** merge: print `len()` before and after, pass `validate=` and `indicator=True`, then `value_counts()` on `_merge`.
A template to copy into your dataset notebook:

```python
print("left :", len(left_df))
print("right:", len(right_df))

result = pd.merge(
    left_df, right_df,
    on="your_key",
    how="left",
    validate="many_to_one",   # what you EXPECT; pandas errors if you're wrong
    indicator=True,
)

print("after:", len(result))            # left + many_to_one → must equal len(left_df)
print(result["_merge"].value_counts())  # both / left_only / right_only

result[result["_merge"] == "left_only"].head()   # look at rows that didn't match
result = result.drop(columns="_merge")
```

How to read it:

| You see | Means | Do |
|---|---|---|
| rows after = rows before (left + m:1) | nothing duplicated | ✅ |
| `MergeError` | a key you thought unique is not | find dups: `df[df.duplicated("key", keep=False)]` |
| big `left_only` | keys don't match | check spaces / case / int vs str in both key columns |
| small `left_only` | a few real non-matches | look at them, decide keep or drop |

---
## Cheat sheet

| I want to… | Use |
|---|---|
| Stack tables with the same columns | `pd.concat([a, b, c], ignore_index=True)` |
| Remember which table each row came from | `pd.concat(..., keys=[...])` or pass a dict |
| Keep only shared columns when stacking | `pd.concat(..., join="inner")` |
| Add columns by matching a key **column** | `pd.merge(a, b, on="key", how="left")` |
| Key has different names | `left_on="x", right_on="y"` |
| Add columns by matching the **index** | `a.join(b)` / `a.join(b, on="col")` |
| Check keys are unique as expected | `validate="many_to_one"` (etc.) |
| See which rows matched | `indicator=True` → `["_merge"].value_counts()` |
| Name clashing columns | `suffixes=("_l", "_r")` |
| Nearest **earlier** time | `pd.merge_asof(a, b, on="time", by="group")` (both sorted) |
| Every combination | `pd.merge(a, b, how="cross")` |
| What did my cleaning change? | `before.compare(after)` |